# In-silico Perturbation

Knock out glomerular filtration-barrier genes (**NPNT, WT1, MAGI1**) *in silico* across the section and measure the effect on each niche's representation with a **W2 (Sinkhorn) distance** between unperturbed and perturbed embeddings.

> **Requirements:** an NVIDIA GPU, and TERRA installed with the `perturb` and `notebook` extras — `terra-st[perturb,notebook]` (`perturb` adds GeomLoss for the perturbation distances; `notebook` adds `gdown` to download the demo data). See the [installation guide](https://terra-st.readthedocs.io/en/latest/installation.html).

This tutorial runs **end-to-end on a de-identified test kidney section we provide** using the public [`lotfollahi-lab/TERRA-96M`](https://huggingface.co/lotfollahi-lab/TERRA-96M) model. Expensive steps (tokenisation, perturbation) are **cached** locally the first run computes them, later runs (and the other tutorials) reuse the cache.

## 1. Setup

In [ ]:
# Install TERRA with the "perturb" and "notebook" extras: perturb adds GeomLoss
# for the perturbation distances, notebook adds gdown to download the demo
# data. Run this if your environment does not already have them -- e.g. on
# Colab or a fresh venv.
%pip install "terra-st[perturb,notebook]"

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import torch
from datasets import load_from_disk

from terra import download_pretrained
from terra.inference import harmonize_adata, tokenize_adata
from terra.inference import perturb_dataset, embed_dataset, summarize_w2_by_label

sc.settings.set_figure_params(dpi=80, frameon=False)
import logging; logging.basicConfig(level="INFO")  # show TERRA progress

/nfs/team361/sb75/.venvs/terra-test/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/nfs/team361/sb75/.venvs/terra-test/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/nfs/team361/sb75/.venvs/terra-test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


## 2. Configuration

Set the model, the section data file, and a local cache directory. If you host the test-section `.h5ad` on Google Drive, put its file id in `DRIVE_FILE_ID` and it will be downloaded automatically.

In [2]:
MODEL_REPO   = "lotfollahi-lab/TERRA-96M"   # HF model bundle
SECTION      = "K014"
NICHE_KEY    = "NEMO_updated_niche"           # obs column used to group / colour
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

DATA_H5AD    = "K014_full.h5ad"          # local path to the section AnnData
DRIVE_FILE_ID = "11C6z0RZXZhbKzI8SHSRbsA_k1i1Tqsfa"  # optional: gdown fallback

CACHE        = Path("cache"); CACHE.mkdir(exist_ok=True)
TOK_CACHE    = CACHE / f"{SECTION}_tokenized"          # shared across tutorials
print("device:", DEVICE)

device: cuda


## 3. Download the pretrained model (Hugging Face)

`download_pretrained` fetches the model bundle (`model_checkpoint.pt`, `model_config.yaml`, `token_dictionary.pkl`) and returns its local path.

In [3]:
model_dir = download_pretrained(MODEL_REPO)
print("model bundle:", model_dir)

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/lotfollahi-lab/TERRA-96M/revision/main "HTTP/1.1 200 OK"


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

model bundle: /nfs/users/nfs_s/sb75/.cache/huggingface/hub/models--lotfollahi-lab--TERRA-96M/snapshots/f3a0ea8b0827fc9d49bea5de27e80f0cbf0214bc


## 4. Load the section data

The provided test-section `.h5ad` holds raw counts (`X`), spatial coordinates (`obsm['spatial']`) and niche / cell-type labels for the whole test section (123,563 cells).

In [4]:
if not os.path.exists(DATA_H5AD):
    import gdown
    print("downloading section data from Google Drive ...")
    gdown.download(id=DRIVE_FILE_ID, output=DATA_H5AD, quiet=False)

adata = sc.read_h5ad(DATA_H5AD)
if "cell_id" not in adata.obs.columns:
    adata.obs["cell_id"] = adata.obs_names.astype(str)
adata.obs["cell_id"] = adata.obs["cell_id"].astype(str)
print(adata)

AnnData object with n_obs × n_vars = 123563 × 4949
    obs: 'cell_id', 'Study_slice_id', 'NEMO_K_SLICE_ID', 'NEMO_updated_niche', 'NEMO_neighborhood_annotation', 'celltype_simplified', 'celltype_broadest'
    obsm: 'spatial'


## 5. Harmonise & tokenise (cached)

**What:** `harmonize_adata` maps gene symbols to the model's Ensembl vocabulary and filters low-quality cells/genes; `tokenize_adata` builds the gene-token sequences the model consumes.

**Input:** raw-count AnnData.  
**Output:** the tokenised section (one row per cell).

> ⏱️ **Slow step — runs once, then cached.** Tokenising the full section (~123k cells) took **≈ 9 min** in our test run. Tokenisation is CPU-bound (it builds the spatial graph and gene-token sequences), so the time depends on your CPU cores / `nproc`, not the GPU. The result is saved to `cache/`, so re-runs — and the other tutorials — load it instantly. Delete `cache/` to recompute.

In [5]:
adata = harmonize_adata(adata)
print(f"after harmonise: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

if TOK_CACHE.exists():
    print("loading cached tokenised dataset ...")
    tok = load_from_disk(str(TOK_CACHE))
else:
    print("tokenising (first run; will be cached) ...")
    tok = tokenize_adata(adata, model_dir, str(CACHE / "tok_tmp"), nproc=4)
    tok.save_to_disk(str(TOK_CACHE))
tok.set_format("torch", columns=["rel_x_coord", "rel_y_coord", "cell_id",
                                 "gene_tokens", "gene_expr", "n_nonzero_tokens"])
print(f"tokenised cells: {len(tok):,}")

INFO:terra.inference.harmonize:STEP 1: DATA VALIDATION...
INFO:terra.inference.harmonize:Checking that adata.X contains raw counts...
INFO:terra.inference.harmonize:✓ adata.X contains raw counts (integer values).
INFO:terra.inference.harmonize:STEP 2: ADDING ENSEMBL IDS...
INFO:terra.inference.harmonize:Adding ensembl IDs from release 111...
INFO:terra.inference.harmonize:Make sure this ensembl release is aligned with pretraining.
INFO:terra.inference.harmonize:Current ensembl release used for pretraining is 111.
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl111/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl111/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl111/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle
INFO:

## 6. Define & run the knockout

**What:** set the target genes to knock out (NPNT/WT1/MAGI1 glomerular filtration-barrier markers) in each cell **and** its neighbourhood, then run the perturbation.

**Input:** the tokenised section + a `perturb_df` describing the knockout.  
**Output:** `perturbed`  the *affected subset* of cells the knockout actually edits (`return_only_perturbed_cells=True`); everything downstream runs on these.

> ⏱️ **Slow step.** ≈ ** 5 minutes** on an **CPU**. Runs once; save the result to avoid recomputing.

In [6]:
genes = ["ENSG00000168743", "ENSG00000184937", "ENSG00000151276"]  # NPNT, WT1, MAGI1
perturb_df = pd.DataFrame({
    "perturbed_cell_id": "all",
    "perturbed_ensembl_id": np.repeat(genes, 2),
    "perturbation_target": ["cell", "neighborhood"] * len(genes),
    "perturbation_type": "knockout", "foldchange": np.nan,
})
PERT_CACHE = CACHE / f"{SECTION}_perturbed"
if PERT_CACHE.exists():
    perturbed = load_from_disk(str(PERT_CACHE))
else:
    perturbed = perturb_dataset(dataset=tok, perturb_df=perturb_df, model_folder_path=model_dir,
                                nproc=1, return_only_perturbed_cells=True,
                                pad_gene_tokens=True, adjust_positions=True,
                                return_perturbation_flags=True)
    perturbed.save_to_disk(str(PERT_CACHE))
affected = list(perturbed.with_format(None)["cell_id"])
print(f"affected cells: {len(affected):,}  (of {len(tok):,})")

affected cells: 45,045  (of 123,563)


## 7. Build the matching control (same cells)

Subset the tokenised section to **exactly the affected cells, in the same order** as `perturbed`. This is the unperturbed baseline so baseline and perturbed are embedded on the identical cell set.

In [7]:
pos = {c: i for i, c in enumerate(tok.with_format(None)["cell_id"])}
control = tok.select([pos[c] for c in affected])
assert list(control.with_format(None)["cell_id"]) == affected

## 8. Embed control + perturbed (affected cells only)

**What:** run the model to get an embedding for every affected cell, once for the unperturbed (control) tokens and once for the perturbed tokens.

**Input:** the aligned `control` and `perturbed` datasets.  
**Output:** `base` and `pert` embeddings (`cell_emb`, `neighborhood_emb`). Both run only over the affected subset.

> ⏱️ **Slow GPU step.** ≈ ** 17 minutes** on an **H100 80GB**. 

In [8]:
emb_kwargs = dict(model_folder_path=model_dir, emb_layer=None, agg_excluded_genes=None,
                  top_k=None, batch_size=128, include_spatial_cell_emb=True,
                  return_token_embeddings=False, ignore_spc_tokens=True, num_workers=8)
base = embed_dataset(dataset=control,   **emb_kwargs)   # baseline, affected subset
pert = embed_dataset(dataset=perturbed, **emb_kwargs)   # perturbed, affected subset

INFO:terra.inference.embed:STEP 1: LOADING CONFIG...
INFO:terra.inference.embed:STEP 2: GENERATING EMBEDDINGS...
INFO:terra.utils.helper:Protein-init: DISABLED -- using the default learnable nn.Embedding for gene tokens.
INFO:terra.utils.helper:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2):

/nfs/team361/sb75/.venvs/terra-test/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


INFO:terra.utils.helper:Loaded pretrained target encoder from epoch 3 with msg: <All keys matched successfully>.
INFO:terra.utils.helper:Finished loading checkpoint with read path: /nfs/users/nfs_s/sb75/.cache/huggingface/hub/models--lotfollahi-lab--TERRA-96M/snapshots/f3a0ea8b0827fc9d49bea5de27e80f0cbf0214bc/model_checkpoint.pt.


0it [00:00, ?it/s]/nfs/team361/sb75/.venvs/terra-test/lib/python3.10/site-packages/terra/inference/embed.py:265: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
352it [10:19,  1.76s/it]


INFO:terra.inference.embed:STEP 1: LOADING CONFIG...
INFO:terra.inference.embed:STEP 2: GENERATING EMBEDDINGS...
INFO:terra.utils.helper:Protein-init: DISABLED -- using the default learnable nn.Embedding for gene tokens.
INFO:terra.utils.helper:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2):

352it [10:23,  1.77s/it]


## 9. Score the perturbation per niche (W2)

**What:** for each niche, measure how far its embeddings *moved* between the unperturbed and perturbed states, using a Wasserstein-2 (Sinkhorn) distance.

**Output:** a table of per-niche W2 scores.  
**How to read it:** a larger W2 = the knockout affected that niche more. The **glomerular** niche is expected to score highest, because we knocked out its marker genes that is the biological sanity check / ground truth.

In [9]:
adata_sub = adata[adata.obs["cell_id"].isin(set(affected))].copy()
obs_ids = adata_sub.obs["cell_id"].astype(str)
for k in ["cell_emb", "neighborhood_emb"]:
    b = pd.DataFrame(np.asarray(base[k]), index=affected)
    p = pd.DataFrame(np.asarray(pert[k]), index=affected)
    adata_sub.obsm[k]              = b.reindex(obs_ids).to_numpy(dtype=np.float32)
    adata_sub.obsm[k + "_perturb"] = p.reindex(obs_ids).to_numpy(dtype=np.float32)

df_w2 = summarize_w2_by_label(
    adata_sub, label_key=NICHE_KEY,
    pairs=[("cell_emb", "cell_emb_perturb", "cell_emb"),
           ("neighborhood_emb", "neighborhood_emb_perturb", "neighborhood_emb")],
    agg="mean", sort_by="neighborhood_emb", ascending=False, device=DEVICE)
df_w2 = df_w2[df_w2["n_cells"] > 50]
df_w2

,label,n_cells,cell_emb,neighborhood_emb
0,Glomerular niche,9726,0.001884,0.001757
1,Proximal straight nephron niche,3426,0.000415,0.000328
2,Perivascular niche,3635,0.000790,0.000311
3,Tubular injury niche,231,0.000865,0.000279
4,Lymphoid infiltrate niche,147,0.000686,0.000217
5,Collecting duct intercalated cell-driven niche,5395,0.000475,0.000172
7,Distal nephron niche,2152,0.000295,0.000095
8,Proximal convoluted nephron niche,10540,0.000341,0.000095
9,Loop of Henle thick ascending limb-driven niche,7302,0.000248,0.000087
10,Collecting duct principal cell-driven niche,2282,0.000256,0.000086


## Next steps

- {doc}`spatial_mapping_tutorial` — compute a *per-cell* perturbation score and project it back onto the tissue.
- {doc}`downstream_analysis` — gene-level embeddings, spatial gene-pair scoring, and EMD spatial structure.

**Questions or issues?** Please report them on the [TERRA GitHub repository](https://github.com/Lotfollahi-lab/terra/issues).